In [3]:
import duckdb
import pandas as pd

# se conecta a una base de datos en memoria
con = duckdb.connect(database=':memory:')

# Explorar el esquema de Personas
print("=== PERSONAS ===")
df_personas_schema = con.execute("DESCRIBE SELECT * FROM '../data/raw/personas_censo2024.parquet'").df()
display(df_personas_schema[['column_name', 'column_type']].head(15)) # Mostramos las primeras 15 para no saturar

# Explorar el esquema de Viviendas (debería contener el identificador geográfico base)
print("\n=== VIVIENDAS ===")
df_viviendas_schema = con.execute("DESCRIBE SELECT * FROM '../data/raw/viviendas_censo2024.parquet'").df()
display(df_viviendas_schema[['column_name', 'column_type']].head(15))

# Explorar el esquema de las Manzanas (CSV)
print("\n=== MANZANAS (CSV) ===")
# read_csv_auto para que DuckDB infiera los tipos de datos
df_manzanas_schema = con.execute("DESCRIBE SELECT * FROM read_csv_auto('../data/raw/Base_manzana_entidad_CPV24.csv')").df()
display(df_manzanas_schema[['column_name', 'column_type']].head(15))

=== PERSONAS ===


,column_name,column_type
0,id_vivienda,INTEGER
1,id_hogar,INTEGER
2,id_persona,INTEGER
3,region,INTEGER
4,provincia,INTEGER
5,comuna,INTEGER
6,comuna_bajo_umbral,INTEGER
7,area,INTEGER
8,tipo_operativo,INTEGER
9,sexo,INTEGER



=== VIVIENDAS ===


,column_name,column_type
0,id_vivienda,INTEGER
1,region,INTEGER
2,provincia,INTEGER
3,comuna,INTEGER
4,comuna_bajo_umbral,INTEGER
5,area,INTEGER
6,tipo_operativo,INTEGER
7,cant_hog,INTEGER
8,cant_per,INTEGER
9,p2_tipo_vivienda,INTEGER



=== MANZANAS (CSV) ===


,column_name,column_type
0,CONTENEDOR_COMUNAL,BIGINT
1,COD_REGION,BIGINT
2,REGION,VARCHAR
3,PROVINCIA,VARCHAR
4,CUT,BIGINT
5,COMUNA,VARCHAR
6,AREA_C,BIGINT
7,MANZENT,BIGINT
8,DISTRITO,VARCHAR
9,COD_DISTRITO,BIGINT


In [4]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')

# DESCRIBE ejecutado directamente y convierte la salida a un DataFrame
query_describe = "DESCRIBE SELECT * FROM read_csv_auto('../data/raw/Base_manzana_entidad_CPV24.csv')"
df_columnas = con.execute(query_describe).df()

# Actualiza el filtro para buscar variables relacionadas a 60 años o más
columnas_candidatas = df_columnas[df_columnas['column_name'].str.contains('edad|60|mayo|pob|adult', case=False, na=False)]

print("Columnas candidatas para Adultos Mayores (60+) en el CSV de Manzanas:")
display(columnas_candidatas[['column_name', 'column_type']])

Columnas candidatas para Adultos Mayores (60+) en el CSV de Manzanas:


,column_name,column_type
26,n_edad_0_5,VARCHAR
27,n_edad_6_13,BIGINT
28,n_edad_14_17,VARCHAR
29,n_edad_18_24,VARCHAR
30,n_edad_25_44,BIGINT
31,n_edad_45_59,BIGINT
32,n_edad_60_mas,BIGINT
33,prom_edad,VARCHAR
111,n_hog_60,BIGINT


In [5]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')

# Query ajustada: read_csv en lugar de read_csv_auto para pasarle el parámetro nullstr
query_rm_adultos = """
SELECT 
    MANZENT AS manzent,
    COMUNA AS comuna,
    n_edad_60_mas AS total_adultos_mayores,
    n_hog_60 AS hogares_con_adultos_mayores
FROM read_csv('../data/raw/Base_manzana_entidad_CPV24.csv', auto_detect=true, nullstr='*')
WHERE COD_REGION = 13       -- Filtramos exclusivamente la Región Metropolitana
  AND n_edad_60_mas > 0     -- Filtra mayores a 0 (y automáticamente descarta los NULLs del asterisco)
"""

# Ejecuta la consulta
df_demografia_rm = con.execute(query_rm_adultos).df()

print("Primeros registros de la Región Metropolitana:")
display(df_demografia_rm.head())
print(f"\nTotal de manzanas con adultos mayores en la RM: {len(df_demografia_rm)}")

# Guarda este dataset procesado
df_demografia_rm.to_parquet('../data/processed/demografia_rm_60mas.parquet')
print("guardado en '../data/processed/demografia_rm_60mas.parquet'")

Primeros registros de la Región Metropolitana:


,manzent,comuna,total_adultos_mayores,hogares_con_adultos_mayores
0,13101011001008,SANTIAGO,49,27
1,13101011001011,SANTIAGO,69,35
2,13101011001012,SANTIAGO,112,65
3,13101011001013,SANTIAGO,49,32
4,13101011001014,SANTIAGO,44,23



Total de manzanas con adultos mayores en la RM: 51802
guardado en '../data/processed/demografia_rm_60mas.parquet'


In [6]:
import os
import pandas as pd
import geopandas as gpd
import json

# Listar archivos de cartografía para identificar el correcto
carto_path = "../data/raw/Cartografia_censo2024_Pais"
archivos_carto = os.listdir(carto_path)
print("=== ARCHIVOS DE CARTOGRAFÍA DISPONIBLES ===")
for f in archivos_carto:
    print(f"- {f}")

# Cargar Establecimientos de Salud (Shapefile)
# Geopandas lee el .shp automáticamente si están todos los archivos (.dbf, .prj, etc) en la misma carpeta
salud_path = "../data/raw/l_910_v1_establecimientos_de_salud_diciembre_2025/l_910_v1_establecimientos_de_salud_diciembre_2025.shp"
try:
    gdf_salud = gpd.read_file(salud_path)
    print(f"\nEstablecimientos de salud cargados: {len(gdf_salud)} registros.")
    # Filtrar solo RM si existe la columna (revisar diccionario luego)
    # gdf_salud = gdf_salud[gdf_salud['REGION'] == '13'] 
except Exception as e:
    print(f"\nError al cargar establecimientos de salud: {e}")

# Cargar Farmacias (JSON)
try:
    with open('../data/raw/farmacias_nacionales.json', 'r', encoding='utf-8') as f:
        data_farmacias = json.load(f)
    df_farmacias = pd.DataFrame(data_farmacias)
    print(f"Farmacias nacionales: {len(df_farmacias)} registros.")
except Exception as e:
    print(f"Error al cargar farmacias: {e}")

=== ARCHIVOS DE CARTOGRAFÍA DISPONIBLES ===
- Cartografia_censo2024_Pais_Aldeas.parquet
- Diccionario_variables_geograficas_CPV24.xlsx
- Cartografia_censo2024_Pais_Manzanas.parquet
- Cartografia_censo2024_Pais_Zonal.parquet
- Cartografia_censo2024_Pais_Distrital.parquet
- Cartografia_censo2024_Pais_Provincial.parquet
- Cartografia_censo2024_Pais_Regional.parquet
- Cartografia_censo2024_Pais_Entidades.parquet
- Cartografia_censo2024_Pais_Comunal.parquet
- Cartografia_censo2024_Pais_Localidades.parquet
- Cartografia_censo2024_Pais_Limite_Urbano.parquet

Establecimientos de salud cargados: 5181 registros.
Farmacias nacionales: 1769 registros.


In [7]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Cargar el parquet espacial
gdf_manzanas = gpd.read_parquet('../data/raw/Cartografia_censo2024_Pais/Cartografia_censo2024_Pais_Manzanas.parquet')

# Filtrar solo Región Metropolitana (COD_REGION == 13)
gdf_manzanas_rm = gdf_manzanas[gdf_manzanas['COD_REGION'] == 13].copy()
print(f"Geometrías RM cargadas: {len(gdf_manzanas_rm)} manzanas.")

# Cargar demografía procesada en el paso anterior
df_demografia = pd.read_parquet('../data/processed/demografia_rm_60mas.parquet')

# Estandarizar las llaves a texto para evitar problemas de merge entre float/int
gdf_manzanas_rm['MANZENT'] = gdf_manzanas_rm['MANZENT'].astype(str).str.split('.').str[0]
df_demografia['manzent'] = df_demografia['manzent'].astype(str).str.split('.').str[0]

# Unir ambos datasets (Inner join para quedarnos solo con manzanas que tienen geometría Y adultos mayores)
gdf_maestro = gdf_manzanas_rm.merge(df_demografia, left_on='MANZENT', right_on='manzent', how='inner')
print(f"Dataset creado: {len(gdf_maestro)} manzanas unificadas.")

# Proyectar a UTM 19S (Metros)
gdf_maestro = gdf_maestro.to_crs(epsg=32719)
# Calcular el punto central de la manzana
gdf_maestro['centroide'] = gdf_maestro.geometry.centroid

# Para filtrar los servicios correctamente se necesita ver cómo se llaman sus columnas
print("\n--- Columnas de Establecimientos de Salud ---")
print(gdf_salud.columns.tolist())

print("\n--- Columnas de Farmacias ---")
print(df_farmacias.columns.tolist())

Geometrías RM cargadas: 66873 manzanas.
Dataset creado: 49813 manzanas unificadas.

--- Columnas de Establecimientos de Salud ---
['ID_ORIG', 'COD_ANT', 'COD_VIG', 'COD_M_ANT', 'COD_M_NUEV', 'COD_REG', 'NOM_REG', 'COD_DEP', 'DEPENDENC', 'PERTENENCI', 'TIPO', 'AMBITO', 'NOMBRE', 'CERTIFI', 'DEP_ADM', 'NIVEL', 'COD_COM', 'NOM_COM', 'VIA', 'NUMERO', 'DIRECCION', 'FONO', 'F_INICIO', 'URGENCIA', 'TIPO_URGE', 'CLAS_SAPU', 'LATITUD', 'LONGITUD', 'PRESTADOR', 'ESTADO', 'COMPLEJIDA', 'TIPO_ATEN', 'CUT_COMUNA', 'CUT_REGION', 'CUT_PROVIN', 'NOM_REGION', 'NOM_PROVIN', 'NOM_COMUNA', 'IR_GOOGLE', 'SIMBOLOGIA', 'geometry']

--- Columnas de Farmacias ---
['fecha', 'local_id', 'local_nombre', 'comuna_nombre', 'localidad_nombre', 'local_direccion', 'funcionamiento_hora_apertura', 'funcionamiento_hora_cierre', 'local_telefono', 'local_lat', 'local_lng', 'funcionamiento_dia', 'fk_region', 'fk_comuna', 'fk_localidad']


In [8]:
import pandas as pd

print("=== ESTABLECIMIENTOS DE SALUD ===")
# Filtrar solo la Región Metropolitana para ver la actualidad (suponiendo que NOM_REGION tiene la RM)
gdf_salud_rm = gdf_salud[gdf_salud['NOM_REGION'].str.contains('METROPOLI', case=False, na=False) | (gdf_salud['COD_REG'] == 13)]

print("Niveles de atención en la RM:")
print(gdf_salud_rm['NIVEL'].value_counts(dropna=False))

print("\nTipos de establecimientos en la RM (Top 15):")
print(gdf_salud_rm['TIPO'].value_counts(dropna=False).head(15))

print("\n=== FARMACIAS ===")
# Buscar farmacias de interés social a nivel nacional o regional
farmacias_sociales = df_farmacias[df_farmacias['local_nombre'].str.contains('POPULAR|COMUNAL|MUNICIPAL|SOLIDARIA', case=False, na=False)]
print(f"Farmacias Populares/Comunales/Solidarias: {len(farmacias_sociales)}")
print("Ejemplos de nombres:")
print(farmacias_sociales['local_nombre'].head(5).tolist())

print("\nCódigos de región en el JSON de farmacias (para saber cuál es la RM):")
print(df_farmacias['fk_region'].value_counts(dropna=False))

=== ESTABLECIMIENTOS DE SALUD ===
Niveles de atención en la RM:
NIVEL
Primario      455
Secundario    276
No Aplica     256
Terciario      68
Pendiente       2
None            1
Name: count, dtype: int64

Tipos de establecimientos en la RM (Top 15):
TIPO
Centro de Salud Privado                                                  235
Centro de Salud Familiar (CESFAM)                                        173
Servicio de Atención Primaria de Urgencia (SAPU)                          92
Clínica                                                                   83
Centro Comunitario de Salud Familiar (CECOSF)                             73
Clínica Dental                                                            63
Laboratorio Clínico                                                       57
Hospital                                                                  46
Centro Comunitario de Salud Mental  (COSAM)                               45
Posta de Salud Rural (PSR)                          

In [9]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Filtrar Establecimientos de Salud de Atención Primaria en la RM
tipos_aps = [
    'Centro de Salud Familiar (CESFAM)', 
    'Centro Comunitario de Salud Familiar (CECOSF)', 
    'Posta de Salud Rural (PSR)',
    'Consultorio General Urbano (CGU)'
]

gdf_salud_aps = gdf_salud_rm[
    (gdf_salud_rm['NIVEL'] == 'Primario') | 
    (gdf_salud_rm['TIPO'].isin(tipos_aps))
].copy()

gdf_salud_aps = gdf_salud_aps.to_crs(epsg=32719)
print(f"Centros de Atención Primaria: {len(gdf_salud_aps)}")

# Procesar y espacializar farmacias con limpieza de coordenadas
df_farmacias['local_lat'] = pd.to_numeric(df_farmacias['local_lat'], errors='coerce')
df_farmacias['local_lng'] = pd.to_numeric(df_farmacias['local_lng'], errors='coerce')

df_farmacias_limpio = df_farmacias.dropna(subset=['local_lat', 'local_lng']).copy()

gdf_farmacias = gpd.GeoDataFrame(
    df_farmacias_limpio, 
    geometry=gpd.points_from_xy(df_farmacias_limpio.local_lng, df_farmacias_limpio.local_lat),
    crs="EPSG:4326" 
)

# Se extrae  la geometría activa sin importar su nombre original
gdf_manzanas_geom = gdf_manzanas_rm.geometry.to_frame().to_crs(epsg=4326)

# Filtrar solo RM
gdf_farmacias_rm = gpd.sjoin(gdf_farmacias, gdf_manzanas_geom, how="inner", predicate="intersects")

gdf_farmacias_rm = gdf_farmacias_rm.to_crs(epsg=32719)
gdf_farmacias_rm = gdf_farmacias_rm.drop_duplicates(subset=['local_id'])
print(f"Farmacias en la RM espacializadas: {len(gdf_farmacias_rm)}")

# Se guardan los datasets procesados para el ruteo
gdf_salud_aps.to_parquet('../data/processed/salud_aps_rm.parquet')
gdf_farmacias_rm.to_parquet('../data/processed/farmacias_rm.parquet')
print("Archivos guardados en 'data/processed/'")

Centros de Atención Primaria: 455
Farmacias en la RM espacializadas: 505
Archivos guardados en 'data/processed/'
